# CYB-5 — Experiment 40 · ONE-CLICK · 3 DistilBERT seeds

**One notebook. No manual copy/paste.** Open in **Google Colab → T4 GPU → Runtime → Run all** and approve Google Drive access.

This notebook safely **reuses** previously trained seeds 42, 123, 2026, automatically **restores** checkpoints from Google Drive after a runtime reset, and **trains only missing seeds**. It produces the mean ± SD table and backs up model files and result ZIP to `MyDrive/CYB5_Experiment40/` automatically.

**Do not delete existing seed folders before verifying the Drive backups.** If you delete a seed whose backup is absent, the notebook must train that seed again. A saved checkpoint alone is not proof that a training run is complete; the notebook verifies its metadata.

**Scope:** This reproduces Experiment 30 on the mostly synthetic dataset. Previously inspected test/original splits are exploratory. This notebook **does not** solve the 88.75% real-email false-positive problem or certify verified phishing labels.

## 1. Install compatible libraries
The `transformers<5` range preserves the APIs used in Experiment 30.

In [ ]:
%pip -q install "transformers>=4.45,<5" "accelerate>=0.27" "pyarrow>=14" "scikit-learn>=1.3" pandas numpy

## 2 · Configuration (no edits required)

The notebook checks GPU availability and uses exactly the original Experiment 40 hyperparameters. Saved checkpoints are not overwritten.

In [ ]:
from pathlib import Path
import os, sys, json, random, time, gc, shutil, zipfile, platform
import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup
import transformers, sklearn

SEEDS = [42, 123, 2026]
BASE_MODEL = 'distilbert/distilbert-base-uncased'
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
FPR_CAP_ON_SYNTHETIC_VALIDATION = 0.05
REQUIRE_GPU = True
COPY_MODELS_TO_DRIVE = True

IS_COLAB = Path('/content').exists()
OUTPUT_DIR = Path('/content/cyb5_exp40_3seeds') if IS_COLAB else Path.cwd()/'cyb5_exp40_3seeds'
DRIVE_DIR = Path('/content/drive/MyDrive/CYB5_Experiment40')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:',sys.version.split()[0], '| PyTorch:',torch.__version__, '| transformers:',transformers.__version__)
print('CUDA:',torch.cuda.is_available(), '| DEVICE:', DEVICE)
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
if REQUIRE_GPU and DEVICE.type != 'cuda':
    raise RuntimeError('Switch Colab runtime to GPU (Runtime > Change runtime type > T4 GPU), then rerun.')
if len(SEEDS)!=len(set(SEEDS)) or len(SEEDS)<3:
    raise ValueError('Use at least three distinct training seeds.')

## 3 · Load the four original files automatically

The notebook mounts Google Drive as needed and loads `train.parquet`, `validation.parquet`, `test.parquet`, and `original_exploratory.parquet` from `MyDrive/CYB5`. It verifies row counts, labels, hashes, and template separation.

In [ ]:
# Experiment 40: correct Colab Drive mount and dataset loading
# All four parquet files already exist in MyDrive/CYB5.
from pathlib import Path
import pandas as pd

FILES = ["train.parquet", "validation.parquet", "test.parquet", "original_exploratory.parquet"]
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/CYB5")

# Mount explicitly BEFORE testing whether files exist.
# Paths on the Google Drive website are not accessible to Colab until it is mounted.
if Path("/content").is_dir():
    try:
        from google.colab import drive
    except ImportError:
        print("Google Colab Drive module unavailable; checking local dataset candidates.")
    else:
        if not all((DRIVE_DATA_DIR / name).is_file() for name in FILES):
            print("Mounting Google Drive to access MyDrive/CYB5 ...")
            drive.mount("/content/drive", force_remount=False)
        else:
            print("Google Drive dataset is already accessible.")

CANDIDATES = [
    DRIVE_DATA_DIR,
    Path("/content/CYB5"),
    Path.cwd() / "CYB5",
    Path.cwd(),
]
DATA_DIR = next(
    (folder for folder in CANDIDATES
     if all((folder / name).is_file() for name in FILES)),
    None
)
if DATA_DIR is None:
    print("\nDataset directory diagnostics:")
    for folder in CANDIDATES:
        found = [name for name in FILES if (folder / name).is_file()]
        print(f"  {folder}: {found}")
    raise FileNotFoundError(
        "Could not find all four Parquet files together. "
        "Google Drive browser shows them in MyDrive/CYB5; "
        "check Colab Drive authorization and rerun this cell."
    )

print("\nDATA_DIR =", DATA_DIR)
sets = {
    key: pd.read_parquet(DATA_DIR / f"{key}.parquet")
    for key in ["train", "validation", "test", "original_exploratory"]
}
expected = {
    "train": 1307,
    "validation": 277,
    "test": 316,
    "original_exploratory": 100,
}
for name, df in sets.items():
    required = {"text", "label", "template_id", "text_hash"}
    if required - set(df.columns):
        raise ValueError(f"{name}: missing columns {required - set(df.columns)}")
    if df[list(required)].isna().any().any():
        raise ValueError(f"{name}: null required field")
    if len(df) != expected[name]:
        raise ValueError(f"{name}: expected {expected[name]}, got {len(df)}")
    if not set(df.label.astype(int).unique()).issubset({0, 1}):
        raise ValueError(f"{name}: unexpected label values")
    if df.text_hash.duplicated().any():
        raise ValueError(f"{name}: duplicate exact text_hash")
    print(f"{name}: n={len(df)} labels={df.label.value_counts().to_dict()} templates={df.template_id.nunique()}")

for left, right in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    if set(sets[left].text_hash) & set(sets[right].text_hash):
        raise RuntimeError(f"Exact-text overlap detected between {left} and {right}")
    if set(sets[left].template_id) & set(sets[right].template_id):
        raise RuntimeError(f"Template overlap detected between {left} and {right}")
print("PASS: all four source datasets loaded; original splits are separated by template/hash.")


## 4. Prepare tokenization and evaluation functions
All seeds share the identical data partitions, tokenizer and hyperparameters. Model initialization, optimizer state, and training shuffle are reset independently for each seed.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
collator = DataCollatorWithPadding(tokenizer=tokenizer,return_tensors='pt')

class EmailDataset(Dataset):
    def __init__(self,df):
        self.labels=df.label.astype(int).tolist()
        self.encoded=tokenizer(df.text.astype(str).tolist(),truncation=True,max_length=MAX_LENGTH)
    def __len__(self): return len(self.labels)
    def __getitem__(self,i):
        d={k:v[i] for k,v in self.encoded.items()}
        d['labels']=self.labels[i]
        return d

datasets={name:EmailDataset(df) for name,df in sets.items()}

def loader(name, shuffle=False, seed=42):
    g=torch.Generator().manual_seed(seed)
    return DataLoader(datasets[name], batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=collator, num_workers=0, generator=g)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True

def predict_scores(model,name):
    dl=loader(name)
    model.eval(); yy=[]; pp=[]; total_seconds=0.0
    with torch.inference_mode():
        for batch in dl:
            inputs={k:v.to(DEVICE) for k,v in batch.items() if k!='labels'}
            if DEVICE.type=='cuda': torch.cuda.synchronize()
            start=time.perf_counter()
            logits=model(**inputs).logits
            if DEVICE.type=='cuda': torch.cuda.synchronize()
            total_seconds+=time.perf_counter()-start
            pp.extend(torch.softmax(logits,dim=-1)[:,1].cpu().tolist())
            yy.extend(batch['labels'].tolist())
    return np.asarray(yy,dtype=int),np.asarray(pp,dtype=float),total_seconds/len(yy)

def calc_metrics(y,p,t):
    predicted=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,predicted,labels=[0,1]).ravel()
    return {'n':int(len(y)),'threshold':float(t),'accuracy':float(accuracy_score(y,predicted)),
            'precision':float(precision_score(y,predicted,zero_division=0)),
            'recall':float(recall_score(y,predicted,zero_division=0)),
            'f1':float(f1_score(y,predicted,zero_division=0)),
            'fpr':float(fp/(fp+tn)) if fp+tn else None,
            'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}

# Threshold is selected exclusively from the synthetic validation partition.
# A grid above 0.90 is included, but no later evaluation set may choose it.
THRESHOLD_GRID = sorted(set([round(float(t),3) for t in np.arange(0.05,1.0,0.01)] + [0.995,0.999]))

def choose_validation_threshold(y,p):
    grid=pd.DataFrame([calc_metrics(y,p,t) for t in THRESHOLD_GRID])
    feasible=grid.loc[grid.fpr<=FPR_CAP_ON_SYNTHETIC_VALIDATION].copy()
    if feasible.empty:
        # Do not pretend that the FPR cap was achieved.
        selection=grid.sort_values(['fpr','f1','threshold'],ascending=[True,False,True]).iloc[0]
        cap_met=False
    else:
        feasible['distance_from_0_5']=(feasible.threshold-0.5).abs()
        selection=feasible.sort_values(['f1','distance_from_0_5','threshold'],ascending=[False,True,True]).iloc[0]
        cap_met=True
    return float(selection.threshold), bool(cap_met), grid

## 5 · Automatically reuse existing seeds; train only missing ones

- If `seed_42`, `seed_123`, or `seed_2026` has a complete, compatible checkpoint and training metadata, reload it and re-evaluate it (no retraining).
- If a local seed folder is missing, first try restoring a complete backup from Drive.
- If a seed is truly missing, train it independently from the original pretrained model.
- If a seed has incomplete/corrupt local artifacts, **archive** that folder rather than overwrite it, then train again.
- Save a complete, verified backup to Drive for each seed.

The checkpoint is selected using validation F1 at 0.50; the operating threshold is selected **only on synthetic validation**. Existing checkpoints retain the threshold they originally selected.

In [ ]:
# EXPERIMENT 40 — ONE-CLICK, IDEMPOTENT TRAINING + RECOVERY
# No manual deletion, no FileExistsError for completed seed folders.
from datetime import datetime, timezone

all_history, all_metrics, seed_metadata = [], [], []
recovered_seeds, newly_trained_seeds, missing_training_history_seeds = [], [], []

REQUIRED = (
    'configuration.json',
    'validation_threshold_grid.csv',
    'best_model/model.safetensors',
    'best_model/config.json',
    'best_model/tokenizer.json',
)


def complete_seed(folder):
    """Weights plus original run metadata: necessary to claim a recovered training run."""
    if not folder.is_dir():
        return False
    return all((folder / rel).is_file() and (folder / rel).stat().st_size > 0
               for rel in REQUIRED)


def compatible_metadata(folder, seed):
    """Refuse to accidentally combine weights from a different experiment."""
    metadata = json.loads((folder / 'configuration.json').read_text(encoding='utf-8'))
    checks = {
        'seed': int(metadata.get('seed', -1)) == seed,
        'base_model': metadata.get('base_model') == BASE_MODEL,
        'max_length': int(metadata.get('max_length', -1)) == MAX_LENGTH,
        'epochs_trained': int(metadata.get('epochs_trained', -1)) == EPOCHS,
        'learning_rate': abs(float(metadata.get('learning_rate', -1)) - LR) < 1e-12,
        'batch_size': int(metadata.get('batch_size', -1)) == BATCH_SIZE,
        'pretrained_init': metadata.get('trained_from_original_pretrained_weights') is True,
        'threshold_recorded': 'validation_selected_threshold' in metadata,
        'epoch_recorded': 'best_epoch' in metadata,
    }
    if not all(checks.values()):
        raise RuntimeError(f'Incompatible existing seed {seed}: {checks}. '
                           'Inspect the saved files rather than mixing experiments.')
    return metadata


def keep_backup(src, dst):
    """Add missing backup files, verify byte sizes, never replace conflicting files."""
    dst.mkdir(parents=True, exist_ok=True)
    for path in sorted(src.rglob('*')):
        if not path.is_file():
            continue
        target = dst / path.relative_to(src)
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            if target.stat().st_size != path.stat().st_size:
                # Per-seed metrics may be regenerated; conflicting checkpoint weights must not be overwritten.
                if ('best_model' in path.relative_to(src).parts):
                    raise RuntimeError(f'Drive checkpoint conflict, preserved: {target}')
                shutil.copy2(path, target)
        else:
            shutil.copy2(path, target)
        if target.stat().st_size != path.stat().st_size:
            raise IOError(f'Backup size verification failed: {target}')
    if not complete_seed(dst):
        raise RuntimeError(f'Incomplete Drive backup at {dst}')
    return dst


def restore_from_drive(seed, destination):
    """Restore full snapshot, or older model+metadata layout from previous notebook."""
    snapshot = DRIVE_DIR / 'resume_snapshots' / f'seed_{seed}'
    if complete_seed(snapshot):
        compatible_metadata(snapshot, seed)
        shutil.copytree(snapshot, destination)
        print(f'  RESTORED seed {seed} from Drive snapshot')
        return True
    model_backup = DRIVE_DIR / f'seed_{seed}'
    meta_backup = DRIVE_DIR / f'seed_{seed}_metadata'
    if ((model_backup / 'model.safetensors').is_file()
            and (meta_backup / 'configuration.json').is_file()
            and (meta_backup / 'validation_threshold_grid.csv').is_file()):
        destination.mkdir(parents=True, exist_ok=False)
        shutil.copytree(model_backup, destination / 'best_model')
        for f in meta_backup.iterdir():
            if f.is_file():
                shutil.copy2(f, destination / f.name)
        if complete_seed(destination):
            compatible_metadata(destination, seed)
            print(f'  RESTORED seed {seed} from older Drive backup layout')
            return True
        raise RuntimeError(f'Old Drive backup for seed {seed} is incomplete: {destination}')
    return False


for SEED in SEEDS:
    print('\n' + '=' * 76)
    print('SEED', SEED)
    print('=' * 76)
    seed_dir = OUTPUT_DIR / f'seed_{SEED}'
    if not seed_dir.exists():
        restore_from_drive(SEED, seed_dir)

    if seed_dir.exists() and not complete_seed(seed_dir):
        # If interrupted, never discard a partial checkpoint silently.
        archive_root = OUTPUT_DIR / '_archived_incomplete'
        archive_root.mkdir(parents=True, exist_ok=True)
        timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
        archive = archive_root / f'seed_{SEED}_{timestamp}'
        seed_dir.rename(archive)
        print(f'  Archived INCOMPLETE run (not deleted): {archive}')
        print('  Training a fresh seed; an incomplete folder is not a completed training run.')

    model_dir = seed_dir / 'best_model'
    meta_path = seed_dir / 'configuration.json'
    grid_path = seed_dir / 'validation_threshold_grid.csv'
    history_path = seed_dir / 'training_history.csv'
    metrics_path = seed_dir / 'per_seed_metrics.csv'

    if complete_seed(seed_dir):
        metadata = compatible_metadata(seed_dir, SEED)
        best_epoch = int(metadata['best_epoch'])
        best_f1 = float(metadata['validation_f1_at_0_5'])
        threshold = float(metadata['validation_selected_threshold'])
        cap_met = bool(metadata['validation_fpr_cap_met'])
        recovered_seeds.append(SEED)
        print(f'  REUSING completed seed {SEED} | epoch {best_epoch} | threshold {threshold:.3f}')
        if not history_path.is_file():
            missing_training_history_seeds.append(SEED)
            print('  Note: epoch-by-epoch history was not saved by the previous notebook.')
        model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE)
    else:
        seed_dir.mkdir(parents=True, exist_ok=True)
        seed_everything(SEED)
        train_loader = loader('train', shuffle=True, seed=SEED)
        model = AutoModelForSequenceClassification.from_pretrained(
            BASE_MODEL, num_labels=2,
            id2label={0: 'SAFE', 1: 'PHISHING'},
            label2id={'SAFE': 0, 'PHISHING': 1},
        ).to(DEVICE)
        optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        steps = len(train_loader) * EPOCHS
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=max(1, int(WARMUP_RATIO * steps)),
            num_training_steps=steps,
        )
        best_f1, best_epoch, epoch_rows = -1.0, None, []
        print(f'  TRAINING missing seed {SEED} from pretrained initialization')
        for epoch in range(1, EPOCHS + 1):
            model.train()
            losses = []
            start_epoch = time.perf_counter()
            for batch in train_loader:
                inputs = {k: v.to(DEVICE) for k, v in batch.items()}
                optimizer.zero_grad(set_to_none=True)
                loss = model(**inputs).loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                losses.append(float(loss.detach().item()))
            y_val, p_val, _ = predict_scores(model, 'validation')
            met = calc_metrics(y_val, p_val, 0.5)
            row = {
                'seed': SEED, 'epoch': epoch, 'train_loss': float(np.mean(losses)),
                'validation_f1_at_0_5': met['f1'],
                'validation_fpr_at_0_5': met['fpr'],
                'epoch_seconds': time.perf_counter() - start_epoch,
            }
            epoch_rows.append(row)
            pd.DataFrame(epoch_rows).to_csv(history_path, index=False)
            print(f'  epoch {epoch} | train loss {row["train_loss"]:.5f} | '
                  f'val F1 {met["f1"]:.4f} | val FPR {met["fpr"]:.4f}')
            if met['f1'] > best_f1 + 1e-12:
                best_f1, best_epoch = met['f1'], epoch
                model.save_pretrained(model_dir, safe_serialization=True)
                tokenizer.save_pretrained(model_dir)
        del model, optimizer, scheduler, train_loader
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
        model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE)
        y_val, p_val, _ = predict_scores(model, 'validation')
        threshold, cap_met, grid = choose_validation_threshold(y_val, p_val)
        grid.insert(0, 'seed', SEED)
        grid.to_csv(grid_path, index=False)
        print(f'  Validation-selected threshold: {threshold:.3f} | FPR cap met: {cap_met}')
        metadata = {
            'seed': SEED, 'base_model': BASE_MODEL, 'best_epoch': best_epoch,
            'validation_f1_at_0_5': best_f1,
            'validation_selected_threshold': threshold,
            'validation_fpr_cap_met': cap_met,
            'trained_from_original_pretrained_weights': True,
            'data_dir': str(DATA_DIR), 'max_length': MAX_LENGTH,
            'epochs_trained': EPOCHS, 'learning_rate': LR,
            'batch_size': BATCH_SIZE, 'weight_decay': WEIGHT_DECAY,
        }
        meta_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
        newly_trained_seeds.append(SEED)

    # For recovered and newly trained seeds, evaluate the same fixed data splits.
    per_seed_rows = []
    for split in ('validation', 'test', 'original_exploratory'):
        yy, pp, sec_per_email = predict_scores(model, split)
        row = calc_metrics(yy, pp, threshold)
        row.update({
            'seed': SEED, 'dataset': split,
            'dataset_status': 'synthetic_validation' if split == 'validation'
                              else 'previously_inspected_exploratory',
            'latency_seconds_per_email_model_forward': sec_per_email,
            'selected_epoch': best_epoch, 'validation_fpr_cap_met': cap_met,
        })
        all_metrics.append(row)
        per_seed_rows.append(row)
        print(f'  {split:23s} | F1 {row["f1"]:.4f} | recall {row["recall"]:.4f} '
              f'| FPR {row["fpr"]:.4f} | n={row["n"]}')
    pd.DataFrame(per_seed_rows).to_csv(metrics_path, index=False)
    seed_metadata.append(metadata)
    if history_path.is_file():
        all_history.extend(pd.read_csv(history_path).to_dict('records'))

    if COPY_MODELS_TO_DRIVE:
        if not DRIVE_DIR.parent.is_dir():
            raise RuntimeError('Google Drive is not mounted: cannot preserve model checkpoints.')
        # Existing snapshot is filled in; no model is deleted or silently overwritten.
        snapshot = DRIVE_DIR / 'resume_snapshots' / f'seed_{SEED}'
        keep_backup(seed_dir, snapshot)
        print(f'  DRIVE BACKUP VERIFIED: {snapshot}')

    del model
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

if len(seed_metadata) != len(SEEDS):
    raise RuntimeError('Not all 3 seeds are complete; do not report 3-seed statistics.')
print('\nALL THREE SEEDS VERIFIED')
print('Already trained:', recovered_seeds)
print('Newly trained:', newly_trained_seeds)
if missing_training_history_seeds:
    print('Missing historical epoch logs (not fabricated):', missing_training_history_seeds)


## 6 · Final results: automatic aggregation and Google Drive export

This cell creates a results CSV, a mean ± SD table and a reproducibility manifest, then saves them as a small ZIP to `MyDrive/CYB5_Experiment40/`. It does not put 800+ MB of model weights in the ZIP because those are backed up separately in the `resume_snapshots` folder.

In [ ]:
# EXPERIMENT 40 — EXPORT FINAL METRICS + MANIFEST
history_df = pd.DataFrame(all_history)
metrics_df = pd.DataFrame(all_metrics)
if not history_df.empty:
    history_df.to_csv(OUTPUT_DIR / 'training_history_3seeds.csv', index=False)
metrics_df.to_csv(OUTPUT_DIR / 'metrics_3seeds.csv', index=False)

summary_df = (metrics_df.groupby('dataset', sort=False)
    .agg(seeds=('seed','nunique'), mean_f1=('f1','mean'), std_f1=('f1','std'),
         mean_recall=('recall','mean'), std_recall=('recall','std'),
         mean_fpr=('fpr','mean'), std_fpr=('fpr','std'),
         mean_accuracy=('accuracy','mean'),
         mean_latency_s=('latency_seconds_per_email_model_forward','mean'))
    .reset_index())
summary_df.to_csv(OUTPUT_DIR / 'summary_mean_std.csv', index=False)

print('\n=== THREE-SEED RESULTS: MEAN ± SD ===')
print(summary_df.round(4).to_string(index=False))
print('\n=== PER-SEED RESULTS ===')
print(metrics_df[['seed','dataset','selected_epoch','threshold','f1','recall','fpr','fp','fn',
                  'latency_seconds_per_email_model_forward']].round(4).to_string(index=False))

manifest = {
    'experiment': 40,
    'goal': 'reproducibility of three independent DistilBERT fine-tunings',
    'seeds': SEEDS,
    'pretrained_model': BASE_MODEL,
    'data_protocol': 'fixed template-separated predominantly synthetic train/validation/test',
    'train_count': len(sets['train']),
    'validation_count': len(sets['validation']),
    'test_count': len(sets['test']),
    'original_exploratory_count': len(sets['original_exploratory']),
    'epoch_selection': 'validation F1 at threshold 0.5; earliest exact tie',
    'threshold_selection': 'synthetic validation only, FPR cap <=0.05 if feasible',
    'external_calibration_performed': False,
    'pilot_exp38_used_for_training_or_selection': False,
    'reused_existing_checkpoints': recovered_seeds,
    'newly_trained_seeds': newly_trained_seeds,
    'missing_epoch_history': missing_training_history_seeds,
    'history_complete_for_all_seeds': len(missing_training_history_seeds) == 0,
    'warning': ('Validation is predominantly synthetic, and both test and original_exploratory '
                'have already been inspected. These metrics do not establish real-world phishing '
                'generalization or adversarial robustness.'),
    'software': {'python': platform.python_version(), 'torch': torch.__version__,
                 'transformers': transformers.__version__, 'sklearn': sklearn.__version__},
    'per_seed': seed_metadata,
}
(OUTPUT_DIR / 'experiment40_manifest.json').write_text(
    json.dumps(manifest, indent=2, default=str), encoding='utf-8')

zip_path = OUTPUT_DIR / 'CYB5_Experiment40_3Seed_Results.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUTPUT_DIR.rglob('*')):
        if not f.is_file() or f == zip_path:
            continue
        if 'best_model' in f.relative_to(OUTPUT_DIR).parts:
            continue
        if '_archived_incomplete' in f.relative_to(OUTPUT_DIR).parts:
            continue
        z.write(f, arcname=str(f.relative_to(OUTPUT_DIR)))

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(zip_path, DRIVE_DIR / zip_path.name)
for name in ('metrics_3seeds.csv', 'summary_mean_std.csv', 'experiment40_manifest.json'):
    shutil.copy2(OUTPUT_DIR / name, DRIVE_DIR / name)
print('\nSUCCESS — Saved to Google Drive:', DRIVE_DIR)
print('Results ZIP:', DRIVE_DIR / zip_path.name)
print('Model backups:', DRIVE_DIR / 'resume_snapshots')
print('Checkpoint restoration / reuse:', recovered_seeds)


## 7 · How to know it worked

At the bottom of the last cell you should see:

- `ALL THREE SEEDS VERIFIED`
- A mean ± SD table for validation, synthetic test, and exploratory original messages
- `SUCCESS — Saved to Google Drive: /content/drive/MyDrive/CYB5_Experiment40`

The report files are available in Google Drive under `CYB5_Experiment40`. Do not delete the three seed folders until the notebook has printed all three `DRIVE BACKUP VERIFIED` messages. You may delete duplicate notebook **copies**, but keep your model backups.

**Scientific limitation:** seed reproducibility on mostly synthetic data is not evidence of deployment readiness. For CYB-5, a separately validated real-email evaluation and the common attack protocol remain necessary. Always disclose any AI-assisted email annotation.